In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/q10_rewrite/checkpoints/pre_cell_4.pickle

In [4]:
%%RecordEvent
%%time
### cell 4 ###

# Optimized for cudf: avoid CPU-side Timestamp conversions
osel = (orders.O_ORDERDATE >= '1994-11-01') & (orders.O_ORDERDATE < '1995-02-01')
lsel = lineitem.L_RETURNFLAG == 'R'

# Filter once
forders = orders[osel]
flineitem = lineitem[lsel]

# Chain merges to minimize intermediates
jn3 = (flineitem
       .merge(forders,     left_on='L_ORDERKEY', right_on='O_ORDERKEY')
       .merge(customer,    left_on='O_CUSTKEY',  right_on='C_CUSTKEY')
       .merge(nation,      left_on='C_NATIONKEY',right_on='N_NATIONKEY'))

# Compute TMP and aggregate
jn3['TMP'] = jn3.L_EXTENDEDPRICE * (1.0 - jn3.L_DISCOUNT)
total = (
    jn3
    .groupby(
        ['C_CUSTKEY','C_NAME','C_ACCTBAL','C_PHONE','N_NAME','C_ADDRESS','C_COMMENT'],
        as_index=False,
        sort=False
    )['TMP']
    .sum()
    .sort_values('TMP', ascending=False)
)


CPU times: user 95.9 ms, sys: 56 ms, total: 152 ms
Wall time: 162 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q10/rewritten/o4_mini_high_small/checkpoints/post_cell_4_try_4.pickle